In [ ]:
import sys
sys.path.insert(0, '..')

# Fuzzy Cognitive Map (FCM) Example: Thucydides Trap

This notebook ports the Thucydides Trap model, originally implemented in Mathematica, to the Python `fcm` library. The model explores the dynamic between a rising and a ruling power.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from lib.fcm import FCM

## 1. Define the FCM Structure

In [ ]:
# Node mappings from trap-verts.csv
mapping = {
    1: "FEAR", 2: "usd", 3: "chnd", 4: "geod", 5: "ENT", 6: "uspub", 7: "chnpub", 8: "dipl", 9: "NUKE", 10: "ShrdCult", 11: "INT", 12: "usecon", 13: "chnecon", 14: "econdep", 15: "allyTangle", 16: "shi", 17: "WAR*"
}

# Edge weights based on the 'Dynamic version (Apr2017)' from trap-eds.txt
activation_threshold = 0.5
addjitter = 0.075
strongOrLink = activation_threshold / 2
orLink = strongOrLink
weakOrLink = activation_threshold / 4
andLink = activation_threshold / 3
subAndLink = activation_threshold + 2 * addjitter

trap_edges = [
    (2, 1, subAndLink), (3, 1, subAndLink),
    (1, 2, subAndLink), (1, 3, subAndLink),
    (4, 1, -weakOrLink),
    (5, 6, subAndLink), (6, 5, subAndLink),
    (5, 7, subAndLink), (7, 5, subAndLink),
    (8, 5, -weakOrLink),
    (14, 12, -weakOrLink), (14, 13, -weakOrLink),
    (12, 11, subAndLink), (13, 11, subAndLink),
    (15, 11, weakOrLink + addjitter),
    (17, 17, orLink),
    (1, 17, 1/3), (5, 17, 1/3), (11, 17, 1/3),
    (4, 17, -andLink - addjitter), (9, 17, -andLink - addjitter), (10, 17, -andLink), (15, 17, andLink),
    (16, 17, andLink),
    (10, 5, -weakOrLink),
    (9, 1, weakOrLink),
    (4, 11, -weakOrLink),
    (8, 11, -weakOrLink),
    (5, 11, weakOrLink + addjitter),
    (4, 14, weakOrLink),
    (10, 14, weakOrLink),
    (14, 10, weakOrLink)
]

## 2. Create and Visualize the FCM

In [ ]:
trap_fcm = FCM("Thucydides Trap Model")
trap_fcm.add_weighted_edges_from(trap_edges)
nx.relabel_nodes(trap_fcm, mapping, copy=False)

plt.figure(figsize=(12, 12))
trap_fcm.draw()
plt.show()

## 3. Define and Run Simulation Scenario

In [ ]:
def run_scenario(fcm, active_nodes, title):
    print(f"--- {title} ---")
    initial_vector = FCM.create_initial_vector(fcm, active_nodes)
    mask = np.zeros_like(initial_vector)
    history = fcm.evolve_to_limit(initial_vector, mask)
    print("Final state:")
    print(history[-1])
    return history

# Scenario Description:
# - US maintains a strong defensive posture
# - China is economically dominant
# - US public resentment is high
# - Both sides are economically dependent
# - Nukes are available to both sides
# - Diplomatic channels are open
active_nodes = ["usd", "chnecon", "uspub", "econdep", "NUKE", "dipl"]
history = run_scenario(trap_fcm, active_nodes, "Thucydides Trap Scenario")